In [1]:
!pip install psycopg2 Faker azure.eventhub

  Preparing metadata (setup.py) ... - \ done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.5 MB/s  0:00:00
  DEPRECATION: Building 'psycopg2' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'psycopg2'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for psycopg2: filename=psycopg2-2.9.12-cp312-cp312-linux_x86_64.whl size=168380 sha256=a7868bd67dab56f0da65b6893a1c39636426f31c6245dd4fb980726db4f2cb05
  Stored in directory: /home/trusted-service-user/.cache/pip/wheels/04/74/ff/720d90509a7e28eeecdaf470a74ba12afb31f8857624440482
Successfully built psycopg2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [azure.eventhub]m [azure.eventhub]


# Imports

In [2]:
"""
simulate_trips.py
-----------------
Generates synthetic ride data with a realistic multi-step status lifecycle,
injects light fraud patterns for demo purposes, writes to Postgres, and
streams every status transition to Fabric Eventstream (Custom App endpoint).

Status flow:
    requested -> confirmed -> in_progress -> completed   (happy path)
    requested -> cancelled                                 (no driver found)
    requested -> confirmed -> cancelled                    (rider cancels - fraud-ish)

Requirements:
    pip install psycopg2-binary faker azure-eventhub

Usage:
    python simulate_trips.py
"""


import time
import random
import math
import logging
import json
import psycopg2

from datetime import datetime, timedelta, timezone
from faker import Faker
from azure.eventhub import EventHubProducerClient, EventData

# Logging

In [3]:
# ──────────────────────────────────────────
# LOGGING CONFIGURATION
# ──────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
logger = logging.getLogger(__name__)
fake = Faker("en_IN")

# DB Config

In [4]:
# ──────────────────────────────────────────
# DB CONFIG  — edit these for your environment
# ──────────────────────────────────────────

connection_id = '25b0bb8a-1274-4e99-91ec-94a6ddca6f07' # connection name: "az-postgres-4-ride-system.postgres.database.azure.com;ride_system sumit.chauhan"
connection_credential = notebookutils.connections.getCredential(connection_id)
credential_dict = json.loads(connection_credential['credential'])
user_name = next(item['value'] for item in credential_dict['credentialData'] if item['name'] == 'username')
pwd = next(item['value'] for item in credential_dict['credentialData'] if item['name'] == 'password')

server_name = 'az-postgres-4-ride-system.postgres.database.azure.com'
database_name = 'ride_system'

DB_CONFIG = {
    "host": server_name,
    "port": 5432,
    "dbname": database_name,
    "user": user_name,
    "password": pwd,
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)


INTERVAL_SECONDS = 2


# Event hub

In [5]:
# ──────────────────────────────────────────
# EVENT HUB / EVENTSTREAM CONFIG
# ──────────────────────────────────────────
EVENT_HUB_CONN_STR = "Endpoint=sb://esehsgouilii9k00zfb46k.servicebus.windows.net/;SharedAccessKeyName=key_076f2ed9-a255-41ae-b596-66348ac7bc2d;SharedAccessKey=CPTh/8g6d8gWNDg4qqUkEkRkgFcm0Z2LL+AEhFPGJmM=;EntityPath=esehsgouilii9k00zfb46k_eh"
EVENT_HUB_NAME = "esehsgouilii9k00zfb46k_eh"

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONN_STR,
    eventhub_name=EVENT_HUB_NAME,
)


def send_event(event_dict):
    """Send a single event to Eventstream. Never crash the main loop on failure."""
    try:
        event_dict["event_sent_at"] = datetime.now(tz=timezone.utc).isoformat()
        batch = producer.create_batch()
        batch.add(EventData(json.dumps(event_dict, default=str)))
        producer.send_batch(batch)
    except Exception as e:
        logger.warning("Eventstream send failed | error=%s", e)


# Static Lookup Data

In [6]:
# ──────────────────────────────────────────
# STATIC LOOKUP DATA
# ──────────────────────────────────────────
PAYMENT_METHODS = ["UPI", "Cash", "Card", "Wallet"]

COMMENTS_GOOD = [
    "Great driver!", "Very polite and punctual.", "Smooth ride, good experience.",
    "Clean vehicle, nice driver.", "Reached on time, good service.",
]
COMMENTS_AVG = [
    "Okay experience.", "Average ride.", "Nothing special.",
    "Decent but could be better.", "Fine overall.",
]
COMMENTS_BAD = [
    "Rash driving.", "Late arrival.", "Not polite.",
    "Vehicle was dirty.", "Poor experience.",
]

# 10 famous Ahmedabad destinations
DESTINATIONS = [
    (23.0600, 72.5808),  # Sabarmati Ashram
    (23.1667, 72.5801),  # Adalaj Stepwell
    (23.0062, 72.6015),  # Kankaria Lake
    (23.0270, 72.5811),  # Sidi Saiyyed Mosque
    (23.0248, 72.5872),  # Jama Masjid
    (23.0253, 72.5800),  # Bhadra Fort
    (23.0258, 72.5886),  # Manek Chowk
    (23.0381, 72.5919),  # Hutheesing Jain Temple
    (23.0506, 72.5925),  # Calico Museum of Textiles
    (22.9818, 72.5034),  # Sarkhej Roza
]

# 20 source/pickup points near city center
SOURCES = [
    (23.0211, 72.5606), (23.0264, 72.5621), (23.0371, 72.5532), (23.0105, 72.5714),
    (23.0374, 72.5535), (23.0135, 72.5721), (23.0075, 72.6042), (23.0345, 72.5484),
    (23.0336, 72.5458), (23.0311, 72.5412), (23.0116, 72.5671), (23.0315, 72.5435),
    (23.0298, 72.6011), (23.0161, 72.5936), (23.0197, 72.5448), (23.0426, 72.5712),
    (23.0232, 72.5728), (23.0145, 72.5615), (23.0398, 72.5881), (23.0353, 72.5292),
]

# fraud injection probability (light bias, per design decision)
FRAUD_INJECT_RATE = 0.08          # ~8% of trips lean fraud-like
SPEEDING_FRAUD_RATE = 0.03        # subset: implausible speed
REPEAT_PAIR_FRAUD_RATE = 0.03     # subset: same driver-rider pair forced
CANCEL_ABUSE_FRAUD_RATE = 0.02    # subset: same driver/rider forced to cancel repeatedly


In [7]:
# ──────────────────────────────────────────
# IN-MEMORY FRAUD STATE (kept small & simple)
# ──────────────────────────────────────────
# Tracks recently used (driver_id, rider_id) pairs and recent cancel-prone actors
# so we can deliberately repeat them for fraud demonstration.
recent_pairs = []          # list of (driver_id, rider_id) tuples
cancel_prone_drivers = []  # driver_ids we deliberately make cancel often
cancel_prone_riders = []   # rider_ids we deliberately make cancel often


# Helper functions

In [8]:
# ──────────────────────────────────────────
# HELPERS — availability checks
# ──────────────────────────────────────────
def fetch_available_rider(cur):
    """
    A rider is available if they have no trip currently in
    ('requested', 'confirmed', 'in_progress') — i.e. not mid-trip.
    """
    cur.execute(
        """
        SELECT r.rider_id
        FROM rider r
        WHERE NOT EXISTS (
            SELECT 1 FROM trips t
            WHERE t.rider_id = r.rider_id
              AND t.status IN ('requested', 'confirmed', 'in_progress')
        )
        ORDER BY random()
        LIMIT 1;
        """
    )
    row = cur.fetchone()
    return row[0] if row else None


def fetch_available_driver(cur):
    """
    A driver is available if active=true and has no trip currently in
    ('confirmed', 'in_progress') — i.e. not mid-trip. (A driver CAN be
    matched to a new request even if a past trip is 'requested' with no
    driver assigned yet, since that's not this driver's trip.)
    """
    cur.execute(
        """
        SELECT d.driver_id
        FROM driver d
        WHERE d.is_active = true
          AND NOT EXISTS (
              SELECT 1 FROM trips t
              WHERE t.driver_id = d.driver_id
                AND t.status IN ('confirmed', 'in_progress')
          )
        ORDER BY random()
        LIMIT 1;
        """
    )
    row = cur.fetchone()
    return row[0] if row else None


def calc_distance(lat1, lon1, lat2, lon2):
    """Rough flat-earth distance in km."""
    dlat = (lat2 - lat1) * 111
    dlon = (lon2 - lon1) * 102
    dist = round(math.sqrt(dlat**2 + dlon**2), 2)
    return max(dist, 1.0)


def random_coord(kind):
    pool = SOURCES if kind == "source" else DESTINATIONS
    return random.choice(pool)


def generate_score():
    r = random.random()
    if r < 0.60:
        return 5.0
    elif r < 0.80:
        return 4.0
    elif r < 0.92:
        return 3.0
    elif r < 0.97:
        return 2.0
    else:
        return 1.0


def pick_comment(score):
    if random.random() > 0.70:
        return None
    if score >= 4.0:
        return random.choice(COMMENTS_GOOD)
    elif score == 3.0:
        return random.choice(COMMENTS_AVG)
    else:
        return random.choice(COMMENTS_BAD)


def update_driver_rating(cur, driver_id):
    cur.execute(
        """
        UPDATE driver
        SET rating = (
            SELECT ROUND(AVG(score)::NUMERIC, 1)
            FROM ratings WHERE driver_id = %s
        )
        WHERE driver_id = %s;
        """,
        (driver_id, driver_id),
    )


def build_trip_event(trip_id, rider_id, driver_id, status, fare, driver_profit,
                      distance_km, duration_min, requested_at, confirmed_at,
                      started_at, completed_at):
    return {
        "event_type": "trip",
        "trip_id": str(trip_id),
        "rider_id": str(rider_id) if rider_id else None,
        "driver_id": str(driver_id) if driver_id else None,
        "status": status,
        "fare_to_rider": fare,
        "profit_to_driver": driver_profit,
        "distance_km": distance_km,
        "duration_minutes": duration_min,
        "requested_at": requested_at.isoformat() if requested_at else None,
        "confirmed_at": confirmed_at.isoformat() if confirmed_at else None,
        "started_at": started_at.isoformat() if started_at else None,
        "completed_at": completed_at.isoformat() if completed_at else None,
    }



# Insertion functions

In [9]:
# ──────────────────────────────────────────
# CORE TRIP LIFECYCLE FUNCTION
# ──────────────────────────────────────────
def run_trip_lifecycle(cur,addToPostgres=True,addToEventHub=True):
    """
    Runs one full trip through its lifecycle, writing each transition to
    Postgres and streaming each transition to Eventstream. Returns
    (trip_id, final_status) or None if no rider/driver available.
    """
    now = datetime.now(tz=timezone.utc)

    rider_id = fetch_available_rider(cur)
    if not rider_id:
        logger.warning("No available riders. Skipping this cycle.")
        return None

    pickup_lat, pickup_lon = random_coord("source")
    dest_lat, dest_lon = random_coord("destination")
    distance_km = calc_distance(pickup_lat, pickup_lon, dest_lat, dest_lon)
    fare = round(50 + distance_km * 12, 2)

    requested_at = now

    # ── STEP 1: INSERT as 'requested', no driver yet ──
    cur.execute(
        """
        INSERT INTO trips (
            rider_id, driver_id,
            pickup_lat, pickup_long, destination_lat, destination_long,
            status, fare_to_rider, profit_to_driver,
            distance_km, duration_minutes, requested_at
        )
        VALUES (%s, NULL, %s,%s, %s,%s, 'requested', %s, NULL, %s, NULL, %s)
        RETURNING trip_id;
        """,
        (rider_id, pickup_lat, pickup_lon, dest_lat, dest_lon, fare, distance_km, requested_at),
    )
    trip_id = cur.fetchone()[0]

    logger.info("Trip requested | trip_id=%s rider_id=%s", trip_id, rider_id)
    send_event(build_trip_event(
        trip_id, rider_id, None, "requested", fare, None,
        distance_km, None, requested_at, None, None, None
    ))

    # ── Decide if a driver is found at all (some requests never get matched) ──
    driver_id = fetch_available_driver(cur)

    if not driver_id:
        # no driver available -> cancelled
        cur.execute(
            "UPDATE trips SET status = 'cancelled' WHERE trip_id = %s;",
            (trip_id,),
        )
        logger.info("Trip cancelled (no driver) | trip_id=%s", trip_id)
        send_event(build_trip_event(
            trip_id, rider_id, None, "cancelled", fare, None,
            distance_km, None, requested_at, None, None, None
        ))
        return trip_id, "cancelled"

    # ── decide fraud injection for this trip ──
    inject_fraud = random.random() < FRAUD_INJECT_RATE
    fraud_kind = None
    if inject_fraud:
        fraud_kind = random.choices(
            ["speeding", "repeat_pair", "cancel_abuse"],
            weights=[SPEEDING_FRAUD_RATE, REPEAT_PAIR_FRAUD_RATE, CANCEL_ABUSE_FRAUD_RATE],
            k=1,
        )[0]

    # force a repeated driver-rider pair for collusion-style fraud demo
    if fraud_kind == "repeat_pair" and recent_pairs:
        forced_driver_id, forced_rider_id = random.choice(recent_pairs)
        # only override if that driver is actually free right now
        cur.execute(
            """
            SELECT 1 FROM driver d
            WHERE d.driver_id = %s AND d.is_active = true
              AND NOT EXISTS (
                  SELECT 1 FROM trips t WHERE t.driver_id = d.driver_id
                  AND t.status IN ('confirmed', 'in_progress')
              );
            """,
            (forced_driver_id,),
        )
        if cur.fetchone():
            driver_id = forced_driver_id

    recent_pairs.append((driver_id, rider_id))
    if len(recent_pairs) > 30:
        recent_pairs.pop(0)

    # ── STEP 2: CONFIRM — driver assigned ──
    confirmed_at = requested_at + timedelta(seconds=random.randint(5, 30))
    cur.execute(
        "UPDATE trips SET status = 'confirmed', driver_id = %s, confirmed_at = %s WHERE trip_id = %s;",
        (driver_id, confirmed_at, trip_id),
    )
    logger.info("Trip confirmed | trip_id=%s driver_id=%s", trip_id, driver_id)
    send_event(build_trip_event(
        trip_id, rider_id, driver_id, "confirmed", fare, None,
        distance_km, None, requested_at, confirmed_at, None, None
    ))

    # ── Decide: rider cancels after confirm? (fraud-ish flow) ──
    rider_cancels = (random.random() < 0.05) or (fraud_kind == "cancel_abuse")
    if fraud_kind == "cancel_abuse":
        # bias toward repeating a known cancel-prone rider/driver
        if cancel_prone_riders and random.random() < 0.5:
            rider_cancels = True
        else:
            cancel_prone_riders.append(rider_id)
            cancel_prone_drivers.append(driver_id)
            if len(cancel_prone_riders) > 10:
                cancel_prone_riders.pop(0)
            if len(cancel_prone_drivers) > 10:
                cancel_prone_drivers.pop(0)

    if rider_cancels:
        cur.execute(
            "UPDATE trips SET status = 'cancelled' WHERE trip_id = %s;",
            (trip_id,),
        )
        logger.info("Trip cancelled (after confirm) | trip_id=%s driver_id=%s rider_id=%s",
                    trip_id, driver_id, rider_id)
        send_event(build_trip_event(
            trip_id, rider_id, driver_id, "cancelled", fare, None,
            distance_km, None, requested_at, confirmed_at, None, None
        ))
        return trip_id, "cancelled"

    # ── STEP 3: IN_PROGRESS — pickup happens ──
    started_at = confirmed_at + timedelta(minutes=1 + random.random() * 5)

    duration_min = int(round(distance_km * (4 + random.random() * 3)))

    if fraud_kind == "speeding":
        # force an implausibly short duration for the distance (fraud signal)
        duration_min = max(1, int(distance_km / (random.uniform(1.0, 1.5))))  # very high km/h

    cur.execute(
        "UPDATE trips SET status = 'in_progress', started_at = %s WHERE trip_id = %s;",
        (started_at, trip_id),
    )
    logger.info("Trip in_progress | trip_id=%s", trip_id)
    send_event(build_trip_event(
        trip_id, rider_id, driver_id, "in_progress", fare, None,
        distance_km, duration_min, requested_at, confirmed_at, started_at, None
    ))

    # small chance trip stays stuck in_progress this cycle (simulates ongoing ride)
    # for simplicity in this simulator we complete it within the same run
    driver_profit = round(fare * 0.75, 2)
    completed_at = started_at + timedelta(minutes=duration_min)

    # ── STEP 4: COMPLETED ──
    cur.execute(
        "UPDATE trips SET status = 'completed', profit_to_driver = %s, duration_minutes = %s, completed_at = %s WHERE trip_id = %s;",
        (driver_profit, duration_min, completed_at, trip_id),
    )
    logger.info("Trip completed | trip_id=%s", trip_id)
    send_event(build_trip_event(
        trip_id, rider_id, driver_id, "completed", fare, driver_profit,
        distance_km, duration_min, requested_at, confirmed_at, started_at, completed_at
    ))

    # ── PAYMENT ──
    method = random.choice(PAYMENT_METHODS)
    r = random.random()
    pay_status = "paid" if r < 0.92 else ("failed" if r < 0.96 else "pending")
    paid_at = completed_at + timedelta(seconds=random.randint(0, 60))

    cur.execute(
        """
        INSERT INTO payments (trip_id, amount, method, status, paid_at)
        VALUES (%s, %s, %s, %s, %s);
        """,
        (trip_id, fare, method, pay_status, paid_at),
    )
    logger.info("Payment created | trip_id=%s amount=%.2f method=%s status=%s",
                trip_id, fare, method, pay_status)

    # ── RATING (85% chance) ──
    if random.random() < 0.85:
        score = generate_score()
        comment = pick_comment(score)
        cur.execute(
            """
            INSERT INTO ratings (trip_id, driver_id, score, comment)
            VALUES (%s, %s, %s, %s);
            """,
            (trip_id, driver_id, score, comment),
        )
        update_driver_rating(cur, driver_id)
        logger.info("Rating created | trip_id=%s driver_id=%s score=%.1f", trip_id, driver_id, score)

    return trip_id, "completed"



# Main

In [10]:
# ──────────────────────────────────────────
# MAIN LOOP
# ──────────────────────────────────────────
def main():
    logger.info("Connecting to PostgreSQL...")
    conn = get_connection()
    conn.autocommit = False
    cur = conn.cursor()
    logger.info("Connected. Running one trip lifecycle every %ss.", INTERVAL_SECONDS)

    trip_count = 0
    try:
        while True:
            try:
                result = run_trip_lifecycle(cur)
                conn.commit()
                if result:
                    trip_count += 1
                    trip_id, status = result
                    logger.info("Cycle committed | count=%s trip_id=%s final_status=%s",
                                trip_count, trip_id, status)
            except Exception:
                conn.rollback()
                logger.exception("Failed trip lifecycle. Transaction rolled back.")

            time.sleep(INTERVAL_SECONDS)

    except KeyboardInterrupt:
        logger.info("Stopped by user. Total trip cycles run=%s", trip_count)

    finally:
        cur.close()
        conn.close()
        producer.close()
        logger.info("Shutdown complete.")




In [11]:
main()

2026-06-22 08:14:58 | INFO     | Connecting to PostgreSQL...
2026-06-22 08:14:58 | INFO     | Connected. Running one trip lifecycle every 2s.
2026-06-22 08:14:58 | INFO     | Trip requested | trip_id=ea2f6ca6-535f-4b3d-b74f-16a26b13b7e9 rider_id=2f89eb1b-0d86-492c-a564-8849ee2d5dd4
2026-06-22 08:14:58 | INFO     | Connection state changed: None -> <ConnectionState.START: 0>
2026-06-22 08:14:58 | INFO     | Connection state changed: <ConnectionState.START: 0> -> <ConnectionState.HDR_SENT: 2>
2026-06-22 08:14:58 | INFO     | Connection state changed: <ConnectionState.HDR_SENT: 2> -> <ConnectionState.HDR_SENT: 2>
2026-06-22 08:14:58 | INFO     | Connection state changed: <ConnectionState.HDR_SENT: 2> -> <ConnectionState.OPEN_PIPE: 4>
2026-06-22 08:14:58 | INFO     | Session state changed: <SessionState.UNMAPPED: 0> -> <SessionState.BEGIN_SENT: 1>
2026-06-22 08:14:58 | INFO     | Link state changed: <LinkState.DETACHED: 0> -> <LinkState.ATTACH_SENT: 1>
2026-06-22 08:14:58 | INFO     | Mana